In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,LSTM
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [2]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [3]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [4]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [5]:
SEQ_LEN = 24
HORIZON = 24

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101593, 24, 13)
X_test Shape:  (21733, 24, 13)


In [6]:
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

In [7]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [8]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [ ]:
#dropout 0.2

In [14]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0071 - mae: 0.0581 - val_loss: 0.0026 - val_mae: 0.0381
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0024 - mae: 0.0374 - val_loss: 0.0021 - val_mae: 0.0338
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0018 - val_mae: 0.0309
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0018 - mae: 0.0320 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0018 - mae: 0.0312 - val_loss: 0.0019 - val_mae: 0.0310
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0016 - val_mae: 0.0281
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [15]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1388.15 MW
RMSE: 1958.44 MW
MAPE: 4.38%
R2: 0.91%


In [16]:
model_lstm.save(r'../models/early_rnn.keras')
history_df =pd.DataFrame(history_lstm.history)

history_df.to_csv(r'../log/early_rnn.csv',index=False)

In [ ]:
#0.3

In [12]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0073 - mae: 0.0612 - val_loss: 0.0027 - val_mae: 0.0392
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0027 - mae: 0.0396 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0022 - mae: 0.0356 - val_loss: 0.0018 - val_mae: 0.0318
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0339 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0019 - mae: 0.0323 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0018 - mae: 0.0320 - val_loss: 0.0017 - val_mae: 0.0295
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [13]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1455.97 MW
RMSE: 1999.42 MW
MAPE: 4.67%
R2: 0.90%


In [ ]:
#0.5

In [17]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0089 - mae: 0.0665 - val_loss: 0.0028 - val_mae: 0.0410
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0033 - mae: 0.0440 - val_loss: 0.0025 - val_mae: 0.0384
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0027 - mae: 0.0391 - val_loss: 0.0020 - val_mae: 0.0323
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0025 - mae: 0.0377 - val_loss: 0.0019 - val_mae: 0.0318
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0024 - mae: 0.0368 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0023 - mae: 0.0360 - val_loss: 0.0017 - val_mae: 0.0303
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0018 - val_mae: 0.0308
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [18]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1537.21 MW
RMSE: 2104.62 MW
MAPE: 4.84%
R2: 0.89%
